# 第七节：双摆 Hamilton 系统与正则变换

> 本 Notebook 聚焦一个核心问题：同一个 Hamilton 系统换到另一组正则坐标后，物理轨迹是否保持一致？

## 学习目标

1. 理解 Hamilton 状态变量 $z=(q,p)$；
2. 用自动微分构造 Hamilton 正则方程；
3. 构造并验证一个正则变换；
4. 比较原坐标与新坐标中的数值轨迹；
5. 用能量漂移和坐标一致性评价结果。

## Apple Silicon 运行说明

- 本节故意使用 **CPU + float64**；
- Apple Silicon 的 MPS 不支持 `float64`；
- Hamiltonian 漂移本身通常很小，双精度比 GPU 加速更重要。


# 1. 双摆 Hamiltonian

状态变量为：

$$z=(q_1,q_2,p_1,p_2)^T$$

其中 $q_1,q_2$ 是两根杆相对竖直向下方向的绝对角度，$p_1,p_2$ 是共轭动量。令：

$$\Delta=q_1-q_2$$

Hamiltonian 为：

$$
H(q,p)=
\frac{
m_2l_2^2p_1^2+(m_1+m_2)l_1^2p_2^2
-2m_2l_1l_2\cos\Delta\,p_1p_2
}{
2m_2l_1^2l_2^2(m_1+m_2\sin^2\Delta)
}
-(m_1+m_2)gl_1\cos q_1-m_2gl_2\cos q_2
$$

无阻尼、无外力且 $H$ 不显含时间，因此理论上：

$$\frac{dH}{dt}=0$$


# 2. Hamilton 正则方程

正则方程写为：

$$\dot q_i=\frac{\partial H}{\partial p_i},
\qquad
\dot p_i=-\frac{\partial H}{\partial q_i}$$

向量形式为：

$$\dot z=J\nabla_zH,\qquad
J=\begin{bmatrix}0&I\\-I&0\end{bmatrix}$$

代码使用 PyTorch 自动微分计算 $\nabla_zH$，避免手工展开复杂导数。


# 3. 正则变换

选择第二类生成函数：

$$F_2(q,P)=q_1P_1+(q_2-q_1)P_2$$

得到新变量：

$$Q_1=q_1,\qquad Q_2=q_2-q_1$$

$$P_1=p_1+p_2,\qquad P_2=p_2$$

逆变换为：

$$q_1=Q_1,\qquad q_2=Q_1+Q_2$$

$$p_1=P_1-P_2,\qquad p_2=P_2$$

新 Hamiltonian 为：

$$K(Q,P)=H(Q_1,Q_1+Q_2,P_1-P_2,P_2)$$

正则性通过矩阵条件检查：

$$A^TJA=J$$


# 4. 对比实验与正确结论

两组坐标描述的是同一个物理系统。将新坐标结果映射回原坐标后，应满足：

$$E_z(t)=\|z_{\mathrm{direct}}(t)-z_{\mathrm{mapped}}(t)\|_2\approx0$$

能量漂移定义为：

$$E_H(t)=|H(t)-H(0)|$$

本实验为公平比较，在两组坐标中都使用 RK4。RK4 不是辛积分器，因此不会严格保持 Hamiltonian；减小时间步后，状态误差和能量漂移应下降。


# 5. 实验流程

1. 定义原始 Hamiltonian；
2. 用自动微分计算 Hamilton 方程右端项；
3. 在原始正则变量中积分；
4. 在正则变换后的变量中积分并映射回来；
5. 比较角度、末端轨迹、状态误差与 Hamiltonian 漂移。


# 配套实验代码

下面按实验流程排列完整代码。首次运行保留 `RUN_QUICK = True`，确认流程正确后再运行正式实验。


## 导入库与双摆物理参数


In [ ]:
from __future__ import annotations

import argparse
import csv
from dataclasses import dataclass
from pathlib import Path
from typing import Callable

import matplotlib.pyplot as plt
import numpy as np
import torch


@dataclass
class Parameters:
    m1: float = 1.0
    m2: float = 1.0
    l1: float = 1.0
    l2: float = 1.0
    g: float = 9.81


## 原始正则变量中的 Hamiltonian 与自动微分


In [ ]:
def hamiltonian(z: torch.Tensor, params: Parameters) -> torch.Tensor:
    q1, q2, p1, p2 = z.unbind(dim=-1)
    delta = q1 - q2
    denominator = (
        2.0
        * params.m2
        * params.l1**2
        * params.l2**2
        * (params.m1 + params.m2 * torch.sin(delta) ** 2)
    )
    numerator = (
        params.m2 * params.l2**2 * p1**2
        + (params.m1 + params.m2) * params.l1**2 * p2**2
        - 2.0
        * params.m2
        * params.l1
        * params.l2
        * torch.cos(delta)
        * p1
        * p2
    )
    potential = (
        -(params.m1 + params.m2) * params.g * params.l1 * torch.cos(q1)
        - params.m2 * params.g * params.l2 * torch.cos(q2)
    )
    return numerator / denominator + potential


def hamiltonian_rhs(
    state: np.ndarray,
    energy_function: Callable[[torch.Tensor, Parameters], torch.Tensor],
    params: Parameters,
) -> np.ndarray:
    """Return (d coordinates/dt, d momenta/dt) using automatic differentiation."""
    # Keep this calculation on CPU: MPS does not support float64, while the
    # small Hamiltonian drift measured in this lesson benefits from precision.
    z = torch.tensor(state, dtype=torch.float64, device="cpu", requires_grad=True)
    energy = energy_function(z, params)
    grad_h = torch.autograd.grad(energy, z)[0]
    dzdt = torch.cat((grad_h[2:], -grad_h[:2]))
    return dzdt.detach().numpy()


## 由生成函数构造正则变换


In [ ]:
def original_to_transformed(z: np.ndarray) -> np.ndarray:
    """Q1=q1, Q2=q2-q1, P1=p1+p2, P2=p2."""
    q1, q2, p1, p2 = np.moveaxis(np.asarray(z), -1, 0)
    return np.stack((q1, q2 - q1, p1 + p2, p2), axis=-1)


def transformed_to_original(z_new: np.ndarray) -> np.ndarray:
    """q1=Q1, q2=Q1+Q2, p1=P1-P2, p2=P2."""
    q1, q_relative, p_total, p2 = np.moveaxis(np.asarray(z_new), -1, 0)
    return np.stack((q1, q1 + q_relative, p_total - p2, p2), axis=-1)


def transformed_hamiltonian(z_new: torch.Tensor, params: Parameters) -> torch.Tensor:
    q1, q_relative, p_total, p2 = z_new.unbind(dim=-1)
    z_original = torch.stack((q1, q1 + q_relative, p_total - p2, p2), dim=-1)
    return hamiltonian(z_original, params)


def symplectic_defect() -> float:
    """Check A^T J A = J for z_original = A z_transformed."""
    identity = np.eye(2)
    zero = np.zeros((2, 2))
    symplectic_matrix = np.block([[zero, identity], [-identity, zero]])
    transform = np.array(
        [
            [1.0, 0.0, 0.0, 0.0],
            [1.0, 1.0, 0.0, 0.0],
            [0.0, 0.0, 1.0, -1.0],
            [0.0, 0.0, 0.0, 1.0],
        ]
    )
    return float(
        np.linalg.norm(transform.T @ symplectic_matrix @ transform - symplectic_matrix)
    )


## 用于公平坐标对比的 RK4


In [ ]:
def integrate_rk4(
    initial_state: np.ndarray,
    t_end: float,
    dt: float,
    energy_function: Callable[[torch.Tensor, Parameters], torch.Tensor],
    params: Parameters,
) -> tuple[np.ndarray, np.ndarray]:
    steps = int(np.ceil(t_end / dt))
    times = np.linspace(0.0, t_end, steps + 1)
    states = np.empty((steps + 1, len(initial_state)), dtype=np.float64)
    states[0] = initial_state

    for index in range(steps):
        step = times[index + 1] - times[index]
        state = states[index]
        k1 = hamiltonian_rhs(state, energy_function, params)
        k2 = hamiltonian_rhs(state + 0.5 * step * k1, energy_function, params)
        k3 = hamiltonian_rhs(state + 0.5 * step * k2, energy_function, params)
        k4 = hamiltonian_rhs(state + step * k3, energy_function, params)
        states[index + 1] = state + step * (k1 + 2 * k2 + 2 * k3 + k4) / 6.0
    return times, states


## 误差评价与可视化


In [ ]:
def energy_history(states: np.ndarray, params: Parameters) -> np.ndarray:
    with torch.no_grad():
        tensor = torch.tensor(states, dtype=torch.float64, device="cpu")
        return hamiltonian(tensor, params).numpy()


def cartesian_positions(states: np.ndarray, params: Parameters) -> tuple[np.ndarray, ...]:
    q1, q2 = states[:, 0], states[:, 1]
    x1 = params.l1 * np.sin(q1)
    y1 = -params.l1 * np.cos(q1)
    x2 = x1 + params.l2 * np.sin(q2)
    y2 = y1 - params.l2 * np.cos(q2)
    return x1, y1, x2, y2


def plot_angles(
    times: np.ndarray, direct: np.ndarray, transformed: np.ndarray, output_dir: Path
) -> None:
    fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
    for index, axis in enumerate(axes):
        axis.plot(times, direct[:, index], label="Direct")
        axis.plot(times, transformed[:, index], "--", label="Canonical transform")
        axis.set_ylabel(f"q{index + 1} [rad]")
        axis.grid(alpha=0.3)
        axis.legend()
    axes[-1].set_xlabel("t [s]")
    fig.tight_layout()
    fig.savefig(output_dir / "01_angles.png", dpi=180)
    plt.close(fig)


def plot_tip_trajectory(
    direct: np.ndarray, transformed: np.ndarray, params: Parameters, output_dir: Path
) -> None:
    _, _, x2_direct, y2_direct = cartesian_positions(direct, params)
    _, _, x2_transformed, y2_transformed = cartesian_positions(transformed, params)
    fig, axis = plt.subplots(figsize=(6, 6))
    axis.plot(x2_direct, y2_direct, label="Direct")
    axis.plot(x2_transformed, y2_transformed, "--", label="Canonical transform")
    axis.set(xlabel="x2", ylabel="y2", title="Trajectory of the second mass")
    axis.axis("equal")
    axis.grid(alpha=0.3)
    axis.legend()
    fig.tight_layout()
    fig.savefig(output_dir / "02_tip_trajectory.png", dpi=180)
    plt.close(fig)


def plot_errors(
    times: np.ndarray,
    direct: np.ndarray,
    transformed: np.ndarray,
    params: Parameters,
    output_dir: Path,
) -> None:
    energy_direct = energy_history(direct, params)
    energy_transformed = energy_history(transformed, params)
    state_error = np.linalg.norm(direct - transformed, axis=1)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].semilogy(times, np.abs(energy_direct - energy_direct[0]) + 1e-18, label="Direct")
    axes[0].semilogy(
        times,
        np.abs(energy_transformed - energy_transformed[0]) + 1e-18,
        "--",
        label="Canonical transform",
    )
    axes[0].set(xlabel="t [s]", ylabel="|H(t)-H(0)|", title="Hamiltonian drift")
    axes[0].legend()
    axes[1].semilogy(times, state_error + 1e-18)
    axes[1].set(xlabel="t [s]", ylabel="||z_direct-z_mapped||", title="Coordinate agreement")
    for axis in axes:
        axis.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(output_dir / "03_energy_and_state_error.png", dpi=180)
    plt.close(fig)


def save_metrics(
    direct: np.ndarray,
    transformed: np.ndarray,
    params: Parameters,
    dt: float,
    output_dir: Path,
) -> None:
    energy_direct = energy_history(direct, params)
    energy_transformed = energy_history(transformed, params)
    rows = [
        {
            "dt": dt,
            "symplectic_defect": symplectic_defect(),
            "max_state_error": np.linalg.norm(direct - transformed, axis=1).max(),
            "max_energy_drift_direct": np.abs(energy_direct - energy_direct[0]).max(),
            "max_energy_drift_transformed": np.abs(
                energy_transformed - energy_transformed[0]
            ).max(),
        }
    ]
    with (output_dir / "metrics.csv").open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)
    print({key: f"{value:.3e}" for key, value in rows[0].items()})


## 完整坐标对比实验


In [ ]:
def run_experiment(t_end: float, dt: float, output_name: str) -> None:
    params = Parameters()
    output_dir = Path.cwd() / "results" / output_name
    output_dir.mkdir(parents=True, exist_ok=True)

    # Zero initial angular velocities imply zero initial canonical momenta.
    original_initial = np.array([1.0, 0.2, 0.0, 0.0])
    transformed_initial = original_to_transformed(original_initial)

    print(f"symplectic defect: {symplectic_defect():.3e}")
    print("integrating original canonical variables...")
    times, direct_states = integrate_rk4(
        original_initial, t_end, dt, hamiltonian, params
    )
    print("integrating transformed canonical variables...")
    _, transformed_states = integrate_rk4(
        transformed_initial, t_end, dt, transformed_hamiltonian, params
    )
    mapped_states = transformed_to_original(transformed_states)

    plot_angles(times, direct_states, mapped_states, output_dir)
    plot_tip_trajectory(direct_states, mapped_states, params, output_dir)
    plot_errors(times, direct_states, mapped_states, params, output_dir)
    save_metrics(direct_states, mapped_states, params, dt, output_dir)
    np.savez(
        output_dir / "trajectories.npz",
        times=times,
        direct=direct_states,
        transformed=transformed_states,
        transformed_mapped=mapped_states,
    )
    print(f"results: {output_dir}")


## 运行实验


In [ ]:
RUN_QUICK = True

if RUN_QUICK:
    run_experiment(t_end=1.0, dt=0.02, output_name="section7_double_pendulum_quick")
else:
    run_experiment(t_end=10.0, dt=0.005, output_name="section7_double_pendulum_dt_0p005")
